In [ ]:
import os
from os import path
import pickle
import numpy as np
from tqdm import tqdm

from macaquethings.plotting.default_styles import *
from macaquethings.plotting.util import legend_without_duplicate_labels

import matplotlib.pyplot as plt
import seaborn as sns
from mycolorpy import colorlist as mcp

figure_style()  # set consistent plotting defaults for all figs

In [ ]:
monkey = 'monkeyN'

"""
# --- lfp
if monkey == 'monkeyN':
    spatiotemporal_results_folder = '../spatiotemporal_decoding/results_monkeyN_roi_3_peaktimes_62_112'
    rnn_results_folder = '../rnn_trajectory_decoding/monkeyN_02-01-2026_12:01:27'
    peak_times = spatiotemporal_results_folder.split('_')[-2:]
else:
    rnn_results_folder = '../rnn_trajectory_decoding/01-31-2026_14:04:25'
    spatiotemporal_results_folder = '../spatiotemporal_decoding/results_monkeyF_roi_3_peaktimes_74_120'
    peak_times = spatiotemporal_results_folder.split('_')[-2:]
    
savedir = 'decoding_compare_models_panels'
"""

# --- mua
if monkey == 'monkeyN':
    spatiotemporal_results_folder = '../spatiotemporal_decoding_mua_chan/results_monkeyN_roi_3_peaktimes_62_112'
    # rnn_results_folder = '../rnn_trajectory_decoding/monkeyN_mua_03-16-2026_13:59:39'
    rnn_results_folder = '../rnn_trajectory_decoding/monkeyN_mua_chan04-10-2026_14:04:51'
    peak_times = spatiotemporal_results_folder.split('_')[-2:]
else:
    # rnn_results_folder = '../rnn_trajectory_decoding/monkeyF_mua_03-16-2026_09:49:16'
    rnn_results_folder = '../rnn_trajectory_decoding/monkeyF_mua_chan04-10-2026_15:01:14'
    spatiotemporal_results_folder = '../spatiotemporal_decoding_mua_chan/results_monkeyF_roi_3_peaktimes_74_120'
    peak_times = spatiotemporal_results_folder.split('_')[-2:]


savedir = 'decoding_compare_models_panels_mua_chan'
os.makedirs(savedir, exist_ok=True)
peak_times

In [ ]:
with open(spatiotemporal_results_folder + '/results.pkl', 'rb') as f:
    spatiotemporal_results = pickle.load(f)

with open(rnn_results_folder + '/perf.pkl', 'rb') as f:
    rnn_delta_t = pickle.load(f)

with open(rnn_results_folder + '/perf_abs_t.pkl', 'rb') as f:
    rnn_abs_t = pickle.load(f)   

with open(rnn_results_folder + '/perf_const_t.pkl', 'rb') as f:
    rnn_const_t = pickle.load(f)   

with open(rnn_results_folder + '/perf_shuffle_t.pkl', 'rb') as f:
    rnn_shuffle_t = pickle.load(f)   


In [ ]:
spatiotemporal_results

In [ ]:
performances = {
    'rnn constant (70 - 170ms)': rnn_const_t,
    r'rnn $x t$': rnn_abs_t,
    r'rnn $x {\Delta t}$ ': rnn_delta_t,
    'rnn shuffle time': rnn_shuffle_t,
}


def get_best_bin_scores(scores, times):
    avg_scores = scores.mean(axis=1)
    idx = np.argmax(avg_scores)
    print(f"best time bin: {times[idx]}ms")
    return scores[idx]
    

performances_linear = {
    'linear - bins': get_best_bin_scores(spatiotemporal_results['single bin scores'], spatiotemporal_results['time']),
    'linear - spatiotemporal': spatiotemporal_results['spatiotemporal scores']['spatiotemporal 0.99'],
    'linear - spatiotemporal n_channels': spatiotemporal_results['spatiotemporal scores']['spatiotemporal n_chans'],
    'linear - avg. (70 - 170ms)': spatiotemporal_results['spatiotemporal scores']['feedforward avg.'],
    f"linear - inter-area times ({'ms, '.join(peak_times) + 'ms'})": spatiotemporal_results['spatiotemporal scores']['inter-area peaks'],
}

spatiotemporal_results['spatiotemporal scores'].keys()

In [ ]:
labels = []
perfs = []

for k,v in performances.items():
    labels.append(k)
    perfs.append(v)


labels_lin, perfs_lin = [], []
for k,v in performances_linear.items():
    labels_lin.append(k)
    perfs_lin.append(v)

rnn_colors = mcp.gen_color(cmap="Reds",n=len(labels))
linear_colors = mcp.gen_color(cmap="Greens",n=len(labels_lin))

kind = [1] * len(labels) + [0] * len(labels_lin)
labels = labels + labels_lin
perfs = perfs + perfs_lin
colors = rnn_colors + linear_colors

kind = np.array(kind)
labels = np.array(labels)
perfs = np.array(perfs)
colors = np.array(colors)


In [ ]:

# sort
idx = np.lexsort((perfs.mean(axis=1), kind))
labels = labels[idx]
perfs = perfs[idx]
colors = colors[idx]
    
plt.figure(figsize=(QUARTER_WIDTH, QUARTER_WIDTH))
for i, perf in enumerate(perfs):
    plt.bar(i, perf.mean(), width=.5, color=colors[i], edgecolor='k')
    plt.scatter((np.ones_like(perf) * i) + np.random.random(size=len(perf)) * 0.2 - 0.1 , perf, color='white', zorder=3, s=2, edgecolor='k', linewidth=.5)
plt.xticks(np.arange(len(labels)), labels, rotation=90)
plt.ylabel('accuracy')

plt.savefig(
    path.join(savedir, f"{monkey}_decoding_bars.svg")
)

# Print mean and standard deviation for each bar

In [ ]:
for label, perf in zip(labels, perfs):
    mean_perf = perf.mean()
    std_perf = perf.std()
    print(f"{label.ljust(40)}: mean = {mean_perf:.3f}, std = {std_perf:.3f}")

In [ ]:
perfs